# Statistics — Contoso profitability

Four things that were awkward or impossible in SQL, so they live here.

```
sql/02_validation.sql         count defects on raw data
notebooks/03_cleaning.ipynb   clean, write out
sql/04_analysis.sql           Q1-Q15
notebooks/05_statistics.ipynb you are here
```

1. Simpson's paradox in the discount-band figures
2. Ruling out product mix as the explanation for the Marketplace gap
3. Whether significance testing applies to this data at all
4. What the discount/margin relationship does and doesn't establish

Reads the clean output of `03_cleaning.ipynb`. Nothing is re-cleaned here.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.width", 150)
CLEAN = Path("../data/clean")

f    = pd.read_csv(CLEAN / "fact_sales.csv", parse_dates=["OrderDate"])
prod = pd.read_csv(CLEAN / "dim_product.csv")
chan = pd.read_csv(CLEAN / "dim_channel.csv")

f = (f.merge(prod[["ProductID", "Category"]], on="ProductID", how="left", suffixes=("", "_p"))
       .merge(chan[["ChannelID", "Channel"]], on="ChannelID", how="left", suffixes=("", "_c")))
if "Category_p" in f: f["Category"] = f["Category"].fillna(f.pop("Category_p"))
if "Channel_c"  in f: f["Channel"]  = f["Channel"].fillna(f.pop("Channel_c"))

# Money metrics use completed orders only - cancelled revenue never happened
net = f[f["OrderStatus"] == "Completed"].copy()

BANDS = ["0%", "0-10%", "10-20%", "20-30%", "30-40%"]
net["DiscountBand"] = pd.Categorical(net["DiscountBand"], BANDS, ordered=True)

def margin(g):
    return 100 * g["ProfitAmount"].sum() / g["SalesAmount"].sum()

print(f"net lines {len(net):,}   revenue Rs {net.SalesAmount.sum()/1e7:,.2f} Cr   "
      f"margin {margin(net):.2f}%")

## 1. Simpson's paradox

Margin by discount band across the whole dataset. It should fall steadily as discounts
deepen.

In [ ]:
overall = net.groupby("DiscountBand", observed=True).apply(margin, include_groups=False).round(1)
print("AGGREGATE — margin % by discount band:\n")
print(overall.to_string())

It doesn't. The 20-30% band (8.4%) is marginally better than 10-20% (8.3%), and the
relationship overall looks weaker than it should.

Holding category constant next.

In [ ]:
within = (net.groupby(["Category", "DiscountBand"], observed=True)
             .apply(margin, include_groups=False).unstack().round(1))
print("WITHIN CATEGORY — monotonic in every single row:\n")
print(within.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))
for cat in within.index:
    row = within.loc[cat].dropna()
    ax.plot(row.index.astype(str), row.values, marker="o", linewidth=1.8, label=cat)
ax.plot(overall.index.astype(str), overall.values, "k--", linewidth=3.2,
        marker="s", markersize=9, label="ALL CATEGORIES", zorder=5)
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Discount band"); ax.set_ylabel("Margin %")
ax.set_title("The aggregate line hides what every category is doing")
ax.legend(fontsize=8.5); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Deep discounts happen almost entirely in high-margin categories. Accessories sits at
31% margin and can absorb a 35% discount; Technology at 10% can't. So the deepest band is
populated by the products that lose least, and the mix flattens the curve.

Within every category the relationship is monotonic.

In [ ]:
print("Maximum discount ever applied, by category:\n")
mx = (net.groupby("Category")
         .agg(MaxDiscount=("DiscountPercent", "max"),
              PctAbove20=("DiscountPercent", lambda s: round(100*(s > 0.20).mean(), 2)),
              Lines=("DiscountPercent", "size"))
         .sort_values("MaxDiscount"))
print(mx.to_string())

Technology is never discounted above 20%. Not one line out of roughly 90,000.

That changes the conclusion from "the deep-discount story is incomplete" to "the
deep-discount story can't be right". A policy capping deep discounts cannot touch a
single Technology transaction, so it can't recover any of the 73% of losses sitting
there.

## 2. Is Marketplace really worse, or is it mix?

Marketplace margin is 9.7% against Website's 15.1%. Commission is the obvious
explanation. The alternative is product mix, i.e. it just sells more thin-margin
Technology.

Those lead to different actions, so this needs decomposing before I recommend anything.

In [ ]:
print("Aggregate margin by channel:\n")
print(net.groupby("Channel").apply(margin, include_groups=False).round(1).to_string())

In [ ]:
print("Margin by channel WITHIN each category:\n")
print((net.groupby(["Category", "Channel"], observed=True)
          .apply(margin, include_groups=False).unstack().round(1)).to_string())

In [ ]:
print("Revenue mix — each channel's share by category (%):\n")
mix = net.pivot_table(index="Channel", columns="Category",
                      values="SalesAmount", aggfunc="sum")
print((100 * mix.div(mix.sum(axis=1), axis=0)).round(1).to_string())

The two tables together rule mix out.

The mix is near-identical: Technology is about 63% of revenue in every channel. And
Marketplace trails in all five categories, not just in the total. The gap survives
controlling for category, which makes it channel economics. Commission and higher return
handling are the plausible mechanism.

If the gap had vanished once category was held constant, the recommendation would have
been to fix the product mix instead of renegotiating commission. Same observation,
opposite action.

What I still can't say: `CostAmount` doesn't itemise commission, so the mechanism is
inferred rather than measured. The gap is measured; the cause is reasoned. I also haven't
tested whether marketplace customers later buy direct.

## 3. Where a p-value doesn't apply

This is every transaction for four years, so it's a population rather than a sample. A
p-value asks whether a pattern could have arisen by chance in a sample drawn from a
population where the effect is zero. Measure the whole population and there's no sampling
error to test against. The difference observed is the difference.

In [ ]:
a = net.loc[net.DiscountBand == "10-20%", "ProfitAmount"]
b = net.loc[net.DiscountBand == "20-30%", "ProfitAmount"]
print(f"10-20% band   mean profit/line Rs {a.mean():>8,.0f}   n = {len(a):,}")
print(f"20-30% band   mean profit/line Rs {b.mean():>8,.0f}   n = {len(b):,}")
print(f"difference                     Rs {a.mean()-b.mean():>8,.0f}")
print()
print("A t-test here returns a vanishingly small p-value purely because n is")
print("large. It would tell you nothing the two means don't already say.")

If you treat these four years as a sample of an ongoing process, which is reasonable
given the point is to say something about next year, then an interval does mean
something. The framing has to be stated either way.

In [ ]:
# Bootstrap CI on the loss-line rate, under the "sample of a process" framing
rng = np.random.default_rng(42)
loss = net["LossFlag"].to_numpy()
boot = np.array([rng.choice(loss, size=len(loss), replace=True).mean()
                 for _ in range(400)])
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"Loss-line rate          {100*loss.mean():.2f}%")
print(f"95% bootstrap interval  [{100*lo:.2f}% , {100*hi:.2f}%]")
print(f"interval width          {100*(hi-lo):.2f} percentage points")
print()
print("Narrow, because n is 146,055. Narrow does NOT mean important - it means")
print("precisely measured. Those are different claims and they get confused a lot.")

## 4. What the discount/margin relationship establishes

The observation is that deeper discounts coincide with lower margin.

Part of that is definitional: a discount on a thin markup has to compress margin. That's
arithmetic, not inference.

What isn't established is the counterfactual, i.e. whether those units would have sold at
full price. Two rival explanations survive the data:

| Rival | Why it's plausible |
|---|---|
| reverse causation | slow-moving, low-margin stock gets discounted because it isn't selling, so the discount is the symptom |
| common cause | thin-markup categories are both heavily discounted and structurally unprofitable, with neither causing the other |

Both produce exactly the pattern observed. Separating them needs a controlled price test
or basket-level data, and neither exists here.

## 5. Figures for the findings document

In [ ]:
band = pd.DataFrame({
    "Lines":        net.groupby("DiscountBand", observed=True).size(),
    "LossRatePct":  (100 * net.groupby("DiscountBand", observed=True)["LossFlag"].mean()).round(1),
    "LossValue_Cr": (-net[net.LossFlag].groupby("DiscountBand", observed=True)["ProfitAmount"]
                      .sum() / 1e7).round(2),
    "MarginPct":    net.groupby("DiscountBand", observed=True).apply(margin, include_groups=False).round(1),
})
band["LossSharePct"] = (100 * band.LossValue_Cr / band.LossValue_Cr.sum()).round(1)
print(band[["Lines", "LossRatePct", "LossValue_Cr", "LossSharePct", "MarginPct"]].to_string())

In [ ]:
# The headline cell: category x band, biggest losses first
cell = (net.groupby(["Category", "DiscountBand"], observed=True)
           .agg(Lines=("SalesAmount", "size"),
                Revenue_Cr=("SalesAmount", lambda s: round(s.sum()/1e7, 2)),
                Loss_Cr=("ProfitAmount", lambda s: round(-s[s < 0].sum()/1e7, 2)))
           .reset_index())
total_loss = -net.loc[net.LossFlag, "ProfitAmount"].sum() / 1e7
cell["PctOfAllLoss"] = (100 * cell.Loss_Cr / total_loss).round(1)
print(cell.nlargest(6, "Loss_Cr").to_string(index=False))
print(f"\ntotal loss across all cells: Rs {total_loss:.2f} Cr")

## Carried into docs/findings_and_recommendations.md

| | |
|---|---|
| section 1 | Simpson's paradox, and Technology never discounted above 20% |
| section 2 | mix ruled out, Marketplace trails in all five categories |
| section 3 | population not sample, bootstrap interval if framed as a sample |
| section 4 | the two rival explanations, used as the limitations section |

Next: `sql/04_analysis.sql` if you haven't run it.